# Project 3 — Clustering & Classification
### Data Mining Fundamentals — Complete Implementation

**Dataset:** Bank Marketing dataset (`bank_cleaned.xlsx`)

This notebook is designed to run on Google Colab and uses the modular `src` package from the GitHub repository.

In [3]:
# ============================================================================
# SETUP: Clone Repository from GitHub
# ============================================================================

import os
import sys
import subprocess

# Clone the repository if not already present
REPO_URL = "https://github.com/AHK8213/DataMining_Clustering_Classification.git"  # UPDATE THIS!

if not os.path.exists("DataMining_Clustering_Classification"):
    print("Cloning repository from GitHub...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("Repository cloned successfully!")
else:
    print("Repository already exists.")

# Change to project directory
os.chdir("DataMining_Clustering_Classification")
print(f"Current working directory: {os.getcwd()}")

# Add src to Python path
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath("./src"))
print("Added src to Python path.")

Repository already exists.
Current working directory: /content/DataMining_Clustering_Classification
Added src to Python path.


In [4]:
# ============================================================================
# SETUP: Install Dependencies
# ============================================================================

print("\n" + "=" * 70)
print("INSTALLING DEPENDENCIES")
print("=" * 70)

# Install requirements
!pip install -q numpy pandas scipy scikit-learn
!pip install -q matplotlib seaborn plotly
!pip install -q torch
!pip install -q xgboost
!pip install -q hdbscan scikit-fuzzy
!pip install -q joblib tqdm

print("\nAll dependencies installed successfully!")

# Verify installations
import numpy as np
import pandas as pd
import sklearn
import torch
import xgboost

print(f"\nNumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"XGBoost version: {xgboost.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


INSTALLING DEPENDENCIES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 14.1 MB/s eta 0:00:00

All dependencies installed successfully!

NumPy version: 2.0.2
Pandas version: 2.2.2
Scikit-learn version: 1.6.1
PyTorch version: 2.11.0+cu128
XGBoost version: 3.3.0
CUDA available: True


---

# PART A — Setup & Data Preparation

## A.1 Imports & Configuration

In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os # Added import for os module

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)

# Ensure src is in path
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath("./src"))

# Import all modules from src package
try:
    from src import (
        # Configuration
        RANDOM_STATE,
        NUM_COLS,
        CAT_COLS,
        TARGET_COL,
        K_RANGE,
        DATA_URL,
        DEVICE,
        GPU_AVAILABLE,
        XGB_DEVICE,

        # Utils
        reduce_memory,
        get_sample,
        hopkins_statistic,
        dunn_index,
        compute_clustering_metrics,
        timer,
        cleanup,

        # Data Preparation
        load_data,
        clean_data,
        create_preprocessor,
        fit_pca,
        prepare_data_for_clustering,
        prepare_data_for_classification,

        # Clustering
        ClusteringRunner,
        run_all_clustering,
        OptimalKDeterminer,
        ClusteringComparator,
        determine_optimal_k,
        ClusteringTendency,
        OptimalKAnalyzer,
        ClusterProfiler,

        # Classification
        prepare_classification_data,
        prepare_pre_call_data,
        BaseClassifier,
        create_all_base_classifiers,
        train_and_evaluate_classifier,
        create_neural_network,
        manual_bagging,
        manual_adaboost,
        run_boosting_depth_experiment,
        compute_classification_metrics,
        cross_validate_classifier,
        cross_validate_multiple_classifiers,
        lift_at_k,
        OverfittingAnalyzer,
        PreCallAnalyzer,
        compare_small_vs_full_dataset,

        # Visualization
        create_figure,
        save_figure,
        plot_cluster_pca,
        plot_dendrogram,
        plot_elbow,
        plot_silhouette_scores,
        plot_roc_curves,
        plot_precision_recall_curves,
        plot_confusion_matrices,
        plot_model_comparison,
        plot_training_history,
        plot_feature_importance,

        # Final Summary
        create_clustering_final_table,
        create_classification_final_table,
        compute_extrinsic_evaluation,
        interpret_extrinsic_evaluation,
        generate_final_discussion,
        ProjectSummary,
        create_complete_summary,
    )
    print("All modules imported successfully!")

    # These print statements are moved inside the try block as they depend on imported variables
    print("=" * 70)
    print("PROJECT 3: CLUSTERING & CLASSIFICATION")
    print("=" * 70)
    print(f"RANDOM_STATE: {RANDOM_STATE}")
    print(f"GPU available: {GPU_AVAILABLE} | Device: {DEVICE}")
    print(f"XGBoost device: {XGB_DEVICE}")
    print("=" * 70)

except ImportError as e:
    print(f"Import error: {e}")
    !find src -maxdepth 2 -type f
    print("Please ensure the repository structure is correct:")
    print("  DataMining_Clustering_Classification/")
    print("  ├── src/")
    print("  │   ├── __init__.py")
    print("  │   └── ...")
    print("  └── notebooks/")
    print("      └── this_notebook.ipynb")
    # Print project title even if config is unavailable due to import error
    print("=" * 70)
    print("PROJECT 3: CLUSTERING & CLASSIFICATION (Configuration variables unavailable due to import error)")
    print("=" * 70)


Import error: No module named 'src.clustering'
src/clustering_tendency.py
src/clustering_algorithms.py
src/base_classifiers.py
src/cluster_profiling.py
src/classification_eval.py
src/ensemble_methods.py
src/final_summary.py
src/config.py
src/classification_analysis.py
src/__init__.py
src/optimal_k.py
src/utils.py
src/data_preparation.py
src/__pycache__/clustering_algorithms.cpython-312.pyc
src/__pycache__/__init__.cpython-312.pyc
src/__pycache__/clustering_evaluation.cpython-312.pyc
src/__pycache__/data_preparation.cpython-312.pyc
src/__pycache__/utils.cpython-312.pyc
src/__pycache__/config.cpython-312.pyc
src/neural_network.py
src/classification_prep.py
src/visualization.py
src/clustering_evaluation.py
Please ensure the repository structure is correct:
  DataMining_Clustering_Classification/
  ├── src/
  │   ├── __init__.py
  │   └── ...
  └── notebooks/
      └── this_notebook.ipynb
PROJECT 3: CLUSTERING & CLASSIFICATION (Configuration variables unavailable due to import error)


## A.2 Memory Optimization

In [ ]:
# ============================================================================
# A.2 Memory Optimization
# ============================================================================

print("\n" + "=" * 70)
print("A.2 MEMORY OPTIMIZATION")
print("=" * 70)

def analyze_memory_usage(df):
    """Analyze and report memory usage of dataframe."""
    memory_before = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage before optimization: {memory_before:.2f} MB")

    df_optimized = reduce_memory(df)
    memory_after = df_optimized.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage after optimization: {memory_after:.2f} MB")
    print(f"Reduction: {(1 - memory_after/memory_before) * 100:.1f}%")

    return df_optimized

print("Memory optimization ready.")
print("Note: Using float64 throughout for numerical consistency.")

## A.3 Data Loading & Cleaning

In [ ]:
# ============================================================================
# A.3 Data Loading & Cleaning
# ============================================================================

print("\n" + "=" * 70)
print("A.3 DATA LOADING & CLEANING")
print("=" * 70)

# Load data
df = load_data(verbose=True)
print(f"Raw shape: {df.shape}")

# Apply memory optimization
df = analyze_memory_usage(df)

# Clean data
df = clean_data(df, verbose=True)
print(f"After cleaning: {df.shape}")

# Generate data quality report
print("\nData Quality Report:")
print("-" * 40)
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## A.4 Feature Selection

In [ ]:
# ============================================================================
# A.4 Feature Selection
# ============================================================================

print("\n" + "=" * 70)
print("A.4 FEATURE SELECTION")
print("=" * 70)

print(f"Available columns: {df.columns.tolist()}")
print(f"\nNumerical columns ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Categorical columns ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Target column: {TARGET_COL}")

print("\nFeature Selection Summary:")
print("-" * 40)
print(f"Features retained: {len(NUM_COLS) + len(CAT_COLS)}")
print(f"Features excluded: {df.shape[1] - len(NUM_COLS) - len(CAT_COLS) - 1}")

X_cluster_df = df.drop(columns=[TARGET_COL])
print(f"\nFeature matrix shape: {X_cluster_df.shape}")

## A.5 PCA

In [ ]:
# ============================================================================
# A.5 PCA
# ============================================================================

print("\n" + "=" * 70)
print("A.5 PCA")
print("=" * 70)

# Create preprocessor
preprocessor = create_preprocessor(NUM_COLS, CAT_COLS)

# Fit preprocessor and transform
X_full = preprocessor.fit_transform(X_cluster_df).astype(np.float64)
print(f"Feature matrix shape: {X_full.shape}")
print(f"Feature matrix dtype: {X_full.dtype}")
print(f"Memory usage: {X_full.nbytes / 1024**2:.2f} MB")

# Fit PCA once on full dataset
pca = fit_pca(X_full, verbose=True)
X_pca_full = pca.transform(X_full)

print(f"\nPCA shape: {X_pca_full.shape}")
print(f"Explained variance (first 10 components): {pca.explained_variance_ratio_[:10].sum():.2%}")

## A.6 Tiered Dataset Preparation

In [ ]:
# ============================================================================
# A.6 Tiered Dataset Preparation
# ============================================================================

print("\n" + "=" * 70)
print("A.6 TIERED DATASET PREPARATION")
print("=" * 70)

def create_tiered_datasets(X_full, sizes):
    """Create tiered datasets for scalability experiments."""
    datasets = {}
    n_total = X_full.shape[0]

    for name, fraction in sizes.items():
        n_samples = int(n_total * fraction)
        indices = np.random.RandomState(RANDOM_STATE).choice(
            n_total, n_samples, replace=False
        )
        datasets[name] = X_full[indices]
        print(f"{name}: {n_samples:,} samples ({fraction*100:.0f}%)")

    return datasets

tiered_sizes = {
    'small': 0.05,
    'medium': 0.20,
    'large': 0.50,
    'full': 1.00
}

tiered_datasets = create_tiered_datasets(X_full, tiered_sizes)
print("\nTiered datasets created for scalability experiments.")

---

# PART B — Clustering Experiments

## B.1 Clustering Tendency

In [ ]:
# ============================================================================
# B.1 Clustering Tendency (Hopkins Statistic)
# ============================================================================

print("\n" + "=" * 70)
print("B.1 CLUSTERING TENDENCY")
print("=" * 70)

tendency = ClusteringTendency(X_full, verbose=True)
H = tendency.compute_hopkins()
print(tendency.get_report())

fig = tendency.plot()
plt.show()

## B.2 Optimal K Determination

In [ ]:
# ============================================================================
# B.2 Optimal K Determination
# - Elbow Method (visualization only)
# - Silhouette Score (primary selection)
# ============================================================================

print("\n" + "=" * 70)
print("B.2 OPTIMAL K DETERMINATION")
print("=" * 70)

best_k, best_by_criteria, determiner = determine_optimal_k(
    X_full, k_range=K_RANGE, verbose=True
)

fig = determiner.plot()
plt.show()

print(f"\nOptimal K (majority vote): {best_k}")
print(f"Best by criteria: {best_by_criteria}")

## B.3 Clustering Algorithms

In [ ]:
# ============================================================================
# B.3 Clustering Algorithms
# - Centroid-Based: K-Means, Bisecting K-Means, K-Medoids, K-Median, Kernel K-Means, Fuzzy C-Means
# - Density-Based: DBSCAN, OPTICS, HDBSCAN
# - Hierarchical: Agglomerative (Ward, Single, Complete) + Dendrogram
# - Probabilistic: Gaussian Mixture Model
# ============================================================================

print("\n" + "=" * 70)
print("B.3 CLUSTERING ALGORITHMS")
print("=" * 70)

print("Running all clustering algorithms...")
labels_dict, runtimes = run_all_clustering(X_full, k=best_k, verbose=True)

print(f"\nRan {len(labels_dict)} algorithms")
print(f"Algorithms: {list(labels_dict.keys())}")

# Generate dendrogram for hierarchical clustering
print("\nGenerating dendrogram...")
from scipy.cluster.hierarchy import dendrogram, linkage

fig, ax = plt.subplots(figsize=(12, 6))
linkage_matrix = linkage(X_full[:1000], method='ward')
dendrogram(linkage_matrix, truncate_mode='level', p=5, ax=ax)
ax.set_title("Dendrogram (Ward Linkage, 1000 samples)")
ax.set_xlabel("Sample Index")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

## B.4 Comprehensive Algorithm Comparison

In [ ]:
# ============================================================================
# B.4 Comprehensive Algorithm Comparison
# - Internal Quality: Silhouette, Davies-Bouldin, Dunn
# - Practicality: Runtime
# - Robustness: Noise ARI (multiple levels)
# - Nonlinear Detection: Two-Moons ARI
# - Final Ranking & Best Model Selection
# ============================================================================

print("\n" + "=" * 70)
print("B.4 COMPREHENSIVE ALGORITHM COMPARISON")
print("=" * 70)

comparator = ClusteringComparator(X_full, labels_dict, runtimes, verbose=True)
comparison_df = comparator.compute_metrics_all()

print("\nComparison results:")
print(comparison_df[['n_clusters', 'noise_pct', 'silhouette', 'davies_bouldin', 'dunn', 'runtime_s']].head())

# Holistic best model
best_clustering_model = comparator.holistic_best_model()
print(f"\nHolistically best model: {best_clustering_model}")
print(f"\nMetrics for best model:")
print(comparison_df.loc[best_clustering_model])

# Plot comparison
fig = comparator.plot_comparison()
plt.show()

## B.5 Cluster Profiling

In [ ]:
# ============================================================================
# B.5 Cluster Profiling (Best Model Only)
# - Numerical Analysis: Means, Medians, Distributions
# - Categorical Analysis: Frequencies, Percentages
# - Feature Importance
# - PCA Visualization (All Clusters Shown)
# ============================================================================

print("\n" + "=" * 70)
print("B.5 CLUSTER PROFILING")
print("=" * 70)

best_labels = labels_dict[best_clustering_model]

profiler = ClusterProfiler(
    df.iloc[:len(best_labels)],
    best_labels,
    feature_columns=NUM_COLS + CAT_COLS,
    num_cols=NUM_COLS,
    cat_cols=CAT_COLS,
    verbose=True
)

print(profiler.get_report())

# Plot numerical profiles
fig = profiler.plot_numerical()
plt.show()

# Plot PCA colored by clusters (ALL clusters shown)
fig = profiler.plot_pca(X_pca_full[:len(best_labels)])
plt.show()

## B.6 Feature Removal Analysis

In [ ]:
# ============================================================================
# B.6 Feature Removal (Using Best Algorithm)
# ============================================================================

print("\n" + "=" * 70)
print("B.6 FEATURE REMOVAL ANALYSIS")
print("=" * 70)

# Get the best algorithm's class from labels_dict
from sklearn.cluster import KMeans

def evaluate_feature_removal(X_full, labels, feature_names, k):
    """Evaluate impact of removing each feature."""
    results = []
    base_score = compute_clustering_metrics(X_full, labels)

    for i, name in enumerate(feature_names):
        X_reduced = np.delete(X_full, i, axis=1)
        # Re-cluster with reduced features
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE)
        labels_reduced = model.fit_predict(X_reduced)
        score = compute_clustering_metrics(X_reduced, labels_reduced)

        results.append({
            'feature_removed': name,
            'silhouette_change': score['silhouette'] - base_score['silhouette'],
            'davies_bouldin_change': score['davies_bouldin'] - base_score['davies_bouldin']
        })

    return pd.DataFrame(results).sort_values('silhouette_change')

# Get feature names
feature_names = X_cluster_df.columns.tolist()
print(f"Evaluating impact of removing {len(feature_names)} features...")

feature_removal_results = evaluate_feature_removal(
    X_full, best_labels, feature_names, best_k
)

print("\nTop 5 features whose removal most hurts silhouette score:")
print(feature_removal_results.head())

print("\nTop 5 features whose removal improves silhouette score:")
print(feature_removal_results.tail())

---

# PART C — Classification Experiments

## C.1 Target Selection & Train/Test Split

In [ ]:
# ============================================================================
# C.1 Target Selection & Train/Test Split (Stratified)
# ============================================================================

print("\n" + "=" * 70)
print("C.1 TARGET SELECTION & TRAIN/TEST SPLIT")
print("=" * 70)

X_train, y_train, X_test, y_test, preprocessor_clf = prepare_classification_data(
    df, verbose=True
)

print(f"\nTrain shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Class distribution in train:\n{pd.Series(y_train).value_counts(normalize=True)}")

## C.2 Feature Preprocessing

In [ ]:
# ============================================================================
# C.2 Feature Preprocessing (Scaling, Encoding, Transformation)
# ============================================================================

print("\n" + "=" * 70)
print("C.2 FEATURE PREPROCESSING")
print("=" * 70)

X_train_enc = preprocessor_clf.fit_transform(X_train).astype(np.float64)
X_test_enc = preprocessor_clf.transform(X_test).astype(np.float64)

print(f"Train encoded shape: {X_train_enc.shape}")
print(f"Test encoded shape: {X_test_enc.shape}")
print(f"Encoded dtype: {X_train_enc.dtype}")
print(f"Memory usage (train): {X_train_enc.nbytes / 1024**2:.2f} MB")
print(f"Memory usage (test): {X_test_enc.nbytes / 1024**2:.2f} MB")

print("\nPreprocessing pipeline saved for consistent transformation.")

## C.3 Base Classifiers

In [ ]:
# ============================================================================
# C.3 Base Classifiers (Cross-Validation Primary)
# - Logistic Regression
# - Decision Tree
# - Naive Bayes
# - Neural Network (PyTorch, 2-3 Hidden Layers, ReLU, Sigmoid, Adam, BCE)
# ============================================================================

print("\n" + "=" * 70)
print("C.3 BASE CLASSIFIERS")
print("=" * 70)

classifiers = create_all_base_classifiers(use_gridsearch=True)
classification_results = []

for name, clf in classifiers.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print('='*50)
    metrics = train_and_evaluate_classifier(
        clf, X_train_enc, y_train, X_test_enc, y_test, verbose=True
    )
    classification_results.append(metrics)

classification_df = pd.DataFrame(classification_results)
classification_df = classification_df.set_index('model')

print("\n" + "=" * 50)
print("Base Classifier Results:")
print("=" * 50)
print(classification_df[['accuracy', 'precision', 'recall', 'f1', 'auc_roc']])

### Neural Network (PyTorch)

In [ ]:
# ============================================================================
# Neural Network: PyTorch Feedforward Network
# - 2-3 Hidden Layers, ReLU Activation, Sigmoid Output
# - Adam Optimizer, Binary Cross-Entropy Loss
# - GPU with CPU fallback
# ============================================================================

print("\n" + "-" * 40)
print("Neural Network (PyTorch):")
print("-" * 40)

try:
    nn_model = create_neural_network(
        hidden_dims=[64, 32, 16],
        epochs=50,
        batch_size=64,
        verbose=True
    )

    nn_model.fit(X_train_enc, y_train.values)

    y_pred_nn = nn_model.predict(X_test_enc)
    y_proba_nn = nn_model.predict_proba(X_test_enc)[:, 1]

    nn_metrics = compute_classification_metrics(y_test, y_pred_nn, y_proba_nn)
    nn_metrics['train_time_s'] = getattr(nn_model, 'train_time_', 0)

    classification_df.loc['Neural Network'] = pd.Series(nn_metrics)

    print(f"\nNeural Network Results:")
    print(f"  F1: {nn_metrics['f1']:.3f}")
    print(f"  AUC: {nn_metrics['auc_roc']:.3f}")

    fig = nn_model.plot_training_history()
    plt.show()

except Exception as e:
    print(f"Neural Network not available: {e}")
    print("Skipping neural network.")

## C.4 Ensemble Methods

In [ ]:
# ============================================================================
# C.4 Ensemble Methods
# - Bagging: Random Forest, Manual Bagging
# - Boosting: XGBoost, Manual Boosting + Depth Analysis
# ============================================================================

print("\n" + "=" * 70)
print("C.4 ENSEMBLE METHODS")
print("=" * 70)

# XGBoost
print("\n" + "-" * 40)
print("XGBoost:")
print("-" * 40)

try:
    import xgboost as xgb

    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        use_label_encoder=False,
        eval_metric='logloss',
        device=XGB_DEVICE
    )

    xgb_model.fit(X_train_enc, y_train)
    y_pred_xgb = xgb_model.predict(X_test_enc)
    y_proba_xgb = xgb_model.predict_proba(X_test_enc)[:, 1]

    xgb_metrics = compute_classification_metrics(y_test, y_pred_xgb, y_proba_xgb)
    classification_df.loc['XGBoost'] = pd.Series(xgb_metrics)

    print(f"XGBoost Results:")
    print(f"  F1: {xgb_metrics['f1']:.3f}")
    print(f"  AUC: {xgb_metrics['auc_roc']:.3f}")

except Exception as e:
    print(f"XGBoost not available: {e}")

# Manual Bagging
print("\n" + "-" * 40)
print("Manual Bagging:")
print("-" * 40)

proba_bag, pred_bag = manual_bagging(
    X_train_enc, y_train.values, X_test_enc,
    n_estimators=15,
    verbose=True
)

bag_metrics = compute_classification_metrics(y_test, pred_bag, proba_bag)
classification_df.loc['Manual Bagging'] = pd.Series(bag_metrics)
print(f"\nManual Bagging Results:")
print(f"  F1: {bag_metrics['f1']:.3f}")
print(f"  AUC: {bag_metrics['auc_roc']:.3f}")

# Manual Boosting (AdaBoost)
print("\n" + "-" * 40)
print("Manual AdaBoost:")
print("-" * 40)

proba_boost, pred_boost, models, alphas = manual_adaboost(
    X_train_enc, y_train.values, X_test_enc,
    n_iterations=40,
    base_max_depth=1,
    verbose=True
)

boost_metrics = compute_classification_metrics(y_test, pred_boost, proba_boost)
classification_df.loc['Manual AdaBoost'] = pd.Series(boost_metrics)
print(f"\nManual AdaBoost Results:")
print(f"  F1: {boost_metrics['f1']:.3f}")
print(f"  AUC: {boost_metrics['auc_roc']:.3f}")

### Boosting Depth Experiment

In [ ]:
# ============================================================================
# Manual Boosting Depth Analysis
# ============================================================================

print("\n" + "-" * 40)
print("Manual Boosting Depth Analysis:")
print("-" * 40)

depth_results = run_boosting_depth_experiment(
    X_train_enc, y_train.values, X_test_enc, y_test,
    depths=[1, 2, 3, 5, 7],
    n_iterations=40,
    verbose=True
)

print("\nDepth Experiment Results:")
print(depth_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(depth_results['depth'], depth_results['f1'], 'o-',
             color='seagreen', linewidth=2, markersize=8)
axes[0].set_title("F1 vs. Base-Learner Depth", fontsize=12)
axes[0].set_xlabel("Depth", fontsize=10)
axes[0].set_ylabel("F1 Score", fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(depth_results['depth'], depth_results['train_time_s'], 'o-',
             color='goldenrod', linewidth=2, markersize=8)
axes[1].set_title("Training Time vs. Base-Learner Depth", fontsize=12)
axes[1].set_xlabel("Depth", fontsize=10)
axes[1].set_ylabel("Time (seconds)", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## C.5 Results Comparison

In [ ]:
# ============================================================================
# C.5 Results Comparison (Consolidated Table, All Models)
# ============================================================================

print("\n" + "=" * 70)
print("C.5 RESULTS COMPARISON")
print("=" * 70)

print("\nComplete Classification Results:")
print("=" * 50)
print(classification_df[['accuracy', 'precision', 'recall', 'f1', 'auc_roc']].round(4))

# Best model
best_classifier = classification_df['f1'].idxmax()
best_f1 = classification_df.loc[best_classifier, 'f1']
print(f"\nBest classifier: {best_classifier}")
print(f"Best F1 score: {best_f1:.4f}")

# Plot comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
fig, ax = plt.subplots(figsize=(12, 6))
classification_df[metrics_to_plot].plot(kind='bar', ax=ax)
ax.set_title("Classification Model Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## C.6 Overfitting & Underfitting Analysis

In [ ]:
# ============================================================================
# C.6 Overfitting & Underfitting Analysis
# - Learning Curves
# - Validation Curves
# - Interpretation (Explanations Required)
# ============================================================================

print("\n" + "=" * 70)
print("C.6 OVERFITTING & UNDERFITTING ANALYSIS")
print("=" * 70)

overfitting_results = []

for name, clf in classifiers.items():
    print(f"\n{'='*50}")
    print(f"Analyzing {name}...")
    print('='*50)

    analyzer = OverfittingAnalyzer(
        clf.pipeline if hasattr(clf, 'pipeline') else clf.model,
        X_train_enc, y_train, X_test_enc, y_test,
        model_name=name,
        verbose=True
    )

    gap = analyzer.compute_gap()
    overfitting_results.append({
        'model': name,
        'train_f1': analyzer.train_f1,
        'val_f1': analyzer.val_f1,
        'overfitting_gap': gap,
        'is_overfitting': analyzer.is_overfitting()
    })

    fig = analyzer.plot_learning_curve()
    plt.show()

overfitting_df = pd.DataFrame(overfitting_results)

print("\n" + "=" * 50)
print("Overfitting Summary:")
print("=" * 50)
print(overfitting_df)

# Interpretation
print("\n" + "=" * 50)
print("Interpretation:")
print("=" * 50)
for _, row in overfitting_df.iterrows():
    model = row['model']
    gap = row['overfitting_gap']
    is_overfit = row['is_overfitting']

    if is_overfit:
        print(f"  - {model}: OVERFITTING detected (gap = {gap:.4f})")
        print(f"    Recommendation: Reduce model complexity or add regularization.")
    elif gap > 0.02:
        print(f"  - {model}: Some overfitting (gap = {gap:.4f})")
        print(f"    Recommendation: Monitor validation performance.")
    else:
        print(f"  - {model}: Good generalization (gap = {gap:.4f})")
        print(f"    Recommendation: Model is well-balanced.")

## C.7 Pre-Call Feature Analysis

In [ ]:
# ============================================================================
# C.7 Pre-Call Feature Analysis
# ============================================================================

print("\n" + "=" * 70)
print("C.7 PRE-CALL FEATURE ANALYSIS")
print("=" * 70)

precall_analyzer = PreCallAnalyzer(
    df,
    target_col=TARGET_COL,
    verbose=True
)

precall_results = precall_analyzer.compare_with_xgboost()
print(precall_analyzer.get_report())

# Lift analysis
_, _, _, _, _, data = precall_results.values()
if len(data) == 4:
    X_test_full_enc, X_test_pc_enc, y_test_full, y_test_pc = data

    model_full, model_pc = precall_results['models']
    proba_full = model_full.predict_proba(X_test_full_enc)[:, 1]
    proba_pc = model_pc.predict_proba(X_test_pc_enc)[:, 1]

    y_series = pd.Series(y_test_full)
    lift_full = lift_at_k(y_series, proba_full, k=0.1)
    lift_pc = lift_at_k(y_series, proba_pc, k=0.1)

    print(f"\nLift at 10% (full features): {lift_full:.2f}")
    print(f"Lift at 10% (pre-call only): {lift_pc:.2f}")

    if lift_pc > 1.5:
        print("  -> Pre-call model can effectively prioritize leads.")
    elif lift_pc > 1.2:
        print("  -> Pre-call model has modest prioritization ability.")
    else:
        print("  -> Pre-call model has little prioritization ability.")

## C.8 Small vs Full Dataset Comparison

In [ ]:
# ============================================================================
# C.8 Small vs Full Dataset Comparison
# ============================================================================

print("\n" + "=" * 70)
print("C.8 SMALL VS FULL DATASET COMPARISON")
print("=" * 70)

# Get small dataset
X_small = tiered_datasets['small']
y_small = df.iloc[:X_small.shape[0]][TARGET_COL].values

print(f"Small dataset: {X_small.shape[0]:,} samples")
print(f"Full dataset: {X_full.shape[0]:,} samples")

def compare_dataset_sizes(X_small, X_full, y_small, y_full):
    """Compare model performance on small vs full dataset."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    import time

    results = []

    for name, model_class in [
        ('Logistic Regression', LogisticRegression),
        ('Decision Tree', DecisionTreeClassifier)
    ]:
        # Small dataset
        model = model_class(random_state=RANDOM_STATE)
        start = time.time()
        model.fit(X_small, y_small)
        small_time = time.time() - start

        # Full dataset
        model = model_class(random_state=RANDOM_STATE)
        start = time.time()
        model.fit(X_full, y_full)
        full_time = time.time() - start

        results.append({
            'model': name,
            'small_time': small_time,
            'full_time': full_time,
            'speedup': full_time / small_time if small_time > 0 else 0
        })

    return pd.DataFrame(results)

comparison_df = compare_dataset_sizes(X_small, X_full, y_small, df[TARGET_COL].values)
print("\nRuntime Comparison:")
print(comparison_df.round(4))

---

# PART D — Summary & Conclusions

## D.1 Final Comparison Tables

In [ ]:
# ============================================================================
# D.1 Final Comparison Tables
# - Clustering Ranking Table
# - Classification Ranking Table
# ============================================================================

print("\n" + "=" * 70)
print("D.1 FINAL COMPARISON TABLES")
print("=" * 70)

print("\n" + "=" * 50)
print("Clustering Ranking Table:")
print("=" * 50)

clustering_final = comparison_df.sort_values('silhouette', ascending=False)
display_cols = ['silhouette', 'davies_bouldin', 'dunn', 'runtime_s', 'noise_pct']
print(clustering_final[display_cols].round(4).head(10))

print("\n" + "=" * 50)
print("Classification Ranking Table:")
print("=" * 50)

classification_final = classification_df.sort_values('f1', ascending=False)
display_cols = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
print(classification_final[display_cols].round(4))

## D.2 Extrinsic Evaluation

In [ ]:
# ============================================================================
# D.2 Extrinsic Evaluation (Clustering vs Classification)
# ============================================================================

print("\n" + "=" * 70)
print("D.2 EXTRINSIC EVALUATION")
print("=" * 70)

extrinsic_results = compute_extrinsic_evaluation(
    df,
    best_labels,
    target_col=TARGET_COL,
    algorithm_name=best_clustering_model,
    verbose=True
)

best_f1 = classification_df['f1'].max()
interpretation = interpret_extrinsic_evaluation(extrinsic_results, best_f1)

print("\n" + "=" * 50)
print("Interpretation:")
print("=" * 50)
for key, value in interpretation.items():
    print(f"\n{key.replace('_', ' ').title()}:")
    print(f"  {value}")

## D.3 Final Discussion

In [ ]:
# ============================================================================
# D.3 Final Discussion
# - Practical Recommendations
# - Limitations
# - Future Work
# ============================================================================

print("\n" + "=" * 70)
print("D.3 FINAL DISCUSSION")
print("=" * 70)

clustering_metrics = comparison_df.loc[best_clustering_model].to_dict()
classification_metrics = classification_df.loc[best_classifier].to_dict()

discussion = generate_final_discussion(
    clustering_best=best_clustering_model,
    classification_best=best_classifier,
    clustering_metrics=clustering_metrics,
    classification_metrics=classification_metrics,
    extrinsic_results=extrinsic_results,
    feature_importance=None
)

print(discussion)

# Save discussion
os.makedirs("../reports", exist_ok=True)
with open("../reports/final_discussion.txt", "w") as f:
    f.write(discussion)
print("\nDiscussion saved to: ../reports/final_discussion.txt")

## Project Completion

In [ ]:
# ============================================================================
# Project Completion Summary
# ============================================================================

print("\n" + "=" * 70)
print("PROJECT 3 COMPLETED SUCCESSFULLY!")
print("=" * 70)
print(f"\nDataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Clustering algorithms evaluated: {len(labels_dict)}")
print(f"Classification models evaluated: {len(classification_df)}")
print(f"\nBest clustering algorithm: {best_clustering_model}")
print(f"Best classifier: {best_classifier}")
print(f"\nExtrinsic ARI: {extrinsic_results['ari']:.3f}")
print(f"Extrinsic NMI: {extrinsic_results['nmi']:.3f}")
print("\n" + "=" * 70)
print("All outputs saved to:")
print("  - reports/project_summary.txt")
print("  - reports/final_discussion.txt")
print("  - figures/ (all plots)")
print("=" * 70)